In [ ]:
from experta import *

# تعريف الوقائع
class State(Fact):
    left = Field(tuple)
    right = Field(tuple)
    light = Field(str)
    time = Field(int)
    path = Field(tuple)
    parent = Field(object, default=None)
    node_id = Field(int, default=0)


# النظام الخبير
class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.visited_states = set()
        self.node_counter = 0
        self.tree = {}  # لتخزين شجرة البحث

    @DefFacts()
    def initial_state(self):
        print("\n🚀 بدأ البحث عن الحل...\n")
        yield State(
            left=('me', 'lab', 'worker', 'scientist'),
            right=(),
            light='left',
            time=0,
            path=(),
            parent=None,
            node_id=0
        )

    def is_goal_state(self, new_left, new_right, new_light):
        return (
            set(new_left) == set() and
            set(new_right) == {'me', 'lab', 'worker', 'scientist'} and
            new_light == 'right'
        )

    # قاعدة عبور شخصين من اليسار إلى اليمين
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node_id=MATCH.node_id), salience=10)
    def move_left_to_right(self, left, right, time, path, node_id):
        parent_id = node_id
        persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        left_list = list(left)
        n = len(left_list)

        for i in range(n):
            for j in range(i + 1, n):
                p1, p2 = left_list[i], left_list[j]
                if 'me' not in (p1, p2):
                    continue  # "me" يجب أن يكون موجودًا

                new_left = left_list.copy()
                new_left.remove(p1)
                new_left.remove(p2)
                new_right = list(right) + [p1, p2]

                t = max(persons_times[p1], persons_times[p2])
                total_time = time + t
                if total_time > 17:
                    continue

                new_light = 'right'
                signature = (tuple(sorted(new_left)), tuple(sorted(new_right)), new_light)
                if signature in self.visited_states:
                    continue
                self.visited_states.add(signature)

                self.node_counter += 1
                new_node_id = self.node_counter
                self.tree[new_node_id] = {
                    'parent': parent_id,
                    'left': new_left,
                    'right': new_right,
                    'light': new_light,
                    'time': total_time
                }

                step = f"{p1} and {p2} crossed to right in {t} min"
                new_path = list(path) + [step]

                self.declare(State(
                    left=tuple(new_left),
                    right=tuple(new_right),
                    light=new_light,
                    time=total_time,
                    path=tuple(new_path),
                    parent=parent_id,
                    node_id=new_node_id
                ))

    # قاعدة عودة شخص واحد من اليمين إلى اليسار
    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node_id=MATCH.node_id), salience=9)
    def move_right_to_left(self, left, right, time, path, node_id):
        parent_id = node_id
        persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        right_list = list(right)

        for p in right_list:
            if p != 'me':
                continue  # فقط "me" يرجع بالمصباح

            new_right = right_list.copy()
            new_right.remove(p)
            new_left = list(left) + [p]

            t = persons_times[p]
            total_time = time + t
            if total_time > 17:
                continue

            new_light = 'left'
            signature = (tuple(sorted(new_left)), tuple(sorted(new_right)), new_light)
            if signature in self.visited_states:
                continue
            self.visited_states.add(signature)

            self.node_counter += 1
            new_node_id = self.node_counter
            self.tree[new_node_id] = {
                'parent': parent_id,
                'left': new_left,
                'right': new_right,
                'light': new_light,
                'time': total_time
            }

            step = f"{p} returned to left in {t} min"
            new_path = list(path) + [step]

            self.declare(State(
                left=tuple(new_left),
                right=tuple(new_right),
                light=new_light,
                time=total_time,
                path=tuple(new_path),
                parent=parent_id,
                node_ٍid=new_node_id
            ))

    # قاعدة الهدف
    @Rule(
        State(left=MATCH.left, right=MATCH.right, light='right',
              time=MATCH.time, path=MATCH.path),
        TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
        salience=100
    )
    def goal_reached(self, left, right, time, path):
        print("\n🎉 ✅ تم الوصول إلى الهدف خلال", time, "دقيقة!\n")
        print("🪄 خطوات الحل:")
        for idx, step in enumerate(path, 1):
            print(f"{idx}. {step}")

        print("\n🌳 شجرة البحث (DFS):")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}) | "
                  f"Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")

        self.halt()


# تشغيل النظام
engine = BridgeExpertSystem()
engine.reset()
engine.run()



🚀 بدأ البحث عن الحل...

